In [1]:
pip uninstall torch torchvision torchaudio

Found existing installation: torch 2.11.0
Uninstalling torch-2.11.0:
  Would remove:
    /usr/local/bin/torchfrtrace
    /usr/local/bin/torchrun
    /usr/local/lib/python3.12/dist-packages/functorch/*
    /usr/local/lib/python3.12/dist-packages/torch-2.11.0.dist-info/*
    /usr/local/lib/python3.12/dist-packages/torch/*
    /usr/local/lib/python3.12/dist-packages/torchgen/*
Proceed (Y/n)? y
  Successfully uninstalled torch-2.11.0
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/torchvision-0.25.0+cu128.dist-info/*
    /usr/local/lib/python3.12/dist-packages/torchvision.libs/libcudart.e8e8b82a.so.12
    /usr/local/lib/python3.12/dist-packages/torchvision.libs/libjpeg.4af9affd.so.8
    /usr/local/lib/python3.12/dist-packages/torchvision.libs/libnvjpeg.8dd2b5e6.so.12
    /usr/local/lib/python3.12/dist-packages/torchvision.libs/libpng16.c2edc9e1.so.16
    /usr/local/lib/python3.12/dist-p

In [3]:
!pip install -U transformers datasets accelerate torch scikit-learn
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130


  Using cached torch-2.11.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (29 kB)
Using cached torch-2.11.0-cp312-cp312-manylinux_2_28_x86_64.whl (530.7 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.
timm 1.0.25 requires torchvision, which is not installed.
Looking in indexes: https://download.pytorch.org/whl/cu130
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 81.1 MB/s eta 0:00:00


In [4]:
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

MODEL_NAME = "CAMeL-Lab/bert-base-arabic-camelbert-mix"
MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 4
LR = 2e-5
OUTPUT_DIR = "./deberta-translation-pedagogy"

dataset = load_dataset(
    "csv",
    data_files="/content/intent_data.csv",
)

dataset = dataset["train"].train_test_split(
    test_size=0.1,
    seed=42,
)

def normalize_labels(batch):
    batch["classes"] = [c.strip().upper() for c in batch["classes"]]
    return batch

dataset = dataset.map(normalize_labels, batched=True)

label2id = {
    "TRANSLATION": 0,
    "TEACHING": 1,
}
id2label = {v: k for k, v in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast = True)

def preprocess(batch):
    return tokenizer(
        batch["utterance"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

dataset = dataset.map(preprocess, batched=True)

def encode_labels(batch):
    batch["labels"] = [label2id[label] for label in batch["classes"]]
    return batch

dataset = dataset.map(encode_labels, batched=True)

dataset = dataset.remove_columns(["utterance", "classes"])
dataset.set_format("torch")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    label2id=label2id,
    id2label=id2label,
    use_safetensors = True
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    logging_steps=50,
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/193 [00:00<?, ? examples/s]

Map:   0%|          | 0/22 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./deberta-translation-pedagogy/tokenizer_config.json',
 './deberta-translation-pedagogy/tokenizer.json')

In [ ]:
from transformers import pipeline
model_path = "./deberta-translation-pedagogy"
classifier = pipeline(
    "text-classification",
    model = model_path,
    tokenizer = model_path,
    return_all_scores = True
)

text = ["ترجم هذه الكلمة الى لغة الاشارة",
        "كيف احكي اكلت هريسة بلغة الاشارة",
        "أعطيني إشارات وسائل الإعلام",
        "بدي اتعلم كل الاشارات المرتبطة بالسوق",
        "ابغى اشارات المرمطة",
        "اريد منك ان تعلمني "]

##### SCORES BEFORE FINETUNING
results = classifier(text)

for i, j in results:
    print(i, "and ", j)

Device set to use cuda:0


{'label': 'TRANSLATION', 'score': 0.5614653825759888} and  {'label': 'PEDAGOGY_CATEGORY', 'score': 0.4385346472263336}
{'label': 'TRANSLATION', 'score': 0.5763416886329651} and  {'label': 'PEDAGOGY_CATEGORY', 'score': 0.4236583709716797}
{'label': 'TRANSLATION', 'score': 0.42183569073677063} and  {'label': 'PEDAGOGY_CATEGORY', 'score': 0.5781643390655518}
{'label': 'TRANSLATION', 'score': 0.5224761366844177} and  {'label': 'PEDAGOGY_CATEGORY', 'score': 0.47752389311790466}
{'label': 'TRANSLATION', 'score': 0.4767117202281952} and  {'label': 'PEDAGOGY_CATEGORY', 'score': 0.5232882499694824}


In [5]:
from transformers import pipeline

model_path = "/content/deberta-translation-pedagogy"
classifier = pipeline(
    "text-classification",
    model = model_path,
    tokenizer = model_path,
    top_k = 2
)

text = ["ترجم هذه الكلمة الى لغة الاشارة",
        "كيف احكي اكلت هريسة بلغة الاشارة",
        "أعطيني إشارات وسائل الإعلام",
        "بدي اتعلم كل الاشارات المرتبطة بالسوق",
        "ابغى اشارات المرمطة",
        "اريد منك ان تعلمني ",
        "بيت جاري",
        "فنجان على الطاولة في غرفة المعيشة"]

######## SCORES AFTER FINETUNING
results = classifier(text)

for i, j in results:
    print(i, "and ", j)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'label': 'TRANSLATION', 'score': 0.9742593765258789} and  {'label': 'TEACHING', 'score': 0.025740671902894974}
{'label': 'TRANSLATION', 'score': 0.9320899248123169} and  {'label': 'TEACHING', 'score': 0.06791006773710251}
{'label': 'TEACHING', 'score': 0.9343942999839783} and  {'label': 'TRANSLATION', 'score': 0.06560571491718292}
{'label': 'TEACHING', 'score': 0.7840833067893982} and  {'label': 'TRANSLATION', 'score': 0.2159167230129242}
{'label': 'TRANSLATION', 'score': 0.629051923751831} and  {'label': 'TEACHING', 'score': 0.37094807624816895}
{'label': 'TRANSLATION', 'score': 0.6288057565689087} and  {'label': 'TEACHING', 'score': 0.3711943030357361}
{'label': 'TRANSLATION', 'score': 0.9320151805877686} and  {'label': 'TEACHING', 'score': 0.06798483431339264}
{'label': 'TRANSLATION', 'score': 0.958747923374176} and  {'label': 'TEACHING', 'score': 0.04125211015343666}
